In [10]:
import numpy as np
import pandas as pd
import requests

In [11]:
BASE_URL = "https://leaderboard-project1-runai-innovation-thibaut.inference.compute.datascience.ch/api/v1"
TEMPLATE_PATH = "./submission.csv"
TARGETS = ["vcd", "tcd", "substrate", "byproduct"]


def get_leaderboard() -> pd.DataFrame:
    leaderboard = requests.get(f"{BASE_URL}/leaderboard").json()
    return pd.json_normalize(leaderboard["entries"])


def submit(df: pd.DataFrame, user: str, token: str = "") -> dict:
    submission = {"predictions": df.to_dict(orient="records")}
    response = requests.post(f"{BASE_URL}/submit", json=submission, auth=(user, token))
    if (response.status_code // 100) != 2:
        raise RuntimeError(f"Unexpected error (HTTP {response.status_code}): {response.text}")
    return response.json()["metrics"]

In [12]:
get_leaderboard()

  user_name  rank  attempt                   timestamp  metrics.vcd_mape  \
0   Team 15     1        8  2026-11-22T08:03:13.449316             6.609   
1   Team 04     2       12  2026-11-22T07:46:46.659185             6.907   
2    Team 7     3        1  2026-11-21T18:06:56.327299             6.745   
3   Team 01     4        6  2026-11-22T08:50:00.730814             6.773   
4    Team 6     5        6  2026-11-22T08:55:15.202915             6.817   
5   Team 23     6       14  2026-11-22T09:27:02.149182             7.201   
6   Team 10     7       97  2026-11-22T09:29:44.875283             7.164   
7   Team 18     8       10  2026-11-21T11:20:35.546502             7.193   
8  TEMPLATE     9       51  2026-11-21T14:24:40.616352             8.037   

   metrics.tcd_mape  metrics.substrate_mape  metrics.byproduct_mape  \
0             6.463                   6.479                   7.457   
1             6.709                   6.633                   6.969   
2             6.649       

In [13]:
template = pd.read_csv(TEMPLATE_PATH, dtype={"batch_id": str})
# Change TEMPLATE with your team name as "Team XX", for example "Team 21" 
metrics = submit(template, user="Team 10")
metrics

{'vcd_mape': 7.197,
 'tcd_mape': 7.025,
 'substrate_mape': 8.925,
 'byproduct_mape': 8.292,
 'mean_mape': 7.86}